# SSW2026 Hands On II Group Project Galactic Context Event

Jessica Lu (UC Berkeley), Matthew Penny (LSU), AND Macy Huston (UC Berkeley)

## Google Colab Usage

**Confirm login account**
* Please make sure to be logged in with the Google account you want to use for the exercises before running the code cells below. You can check by clicking the circular account icon in the top right corner of the colab notebook.

**Working directory**
* Note: The data is shared via a google drive. See the workshop Google Colab instructions to create the shorcut to your Google Drive.

**Running cells**
* Run cells individually by clicking on the triangle on each cell
* For text cells, you can click the down arrow to get to the next cell

**To Restart runtime**
*   Click on Runtime menu item
*   Select Restart session
*   Select Run code cells individually from the top

**To Recreate runtime**
*   Click on Runtime menu item
*   Select Disconnect and Delete runtime
*   Select Run code cells individually from the top

**To Exit:**
*   Close the browser window

# Microlens or Transit Event in Context
## Galactic Center

Once we have detected a microlensing event or a transiting system, we wish to know everything we can about the lens or the host star. This often requires modeling photometry, estimating distances based on color, luminosity, or parallax, and comparing the proper motions to the surrounding stars. Of course, these are indirect probes of the physical mass, distance, luminosity, and spectral type of the lens/host star. To use the observed quanitities to constrain physical properties, we sometimes need to depend on Galactic models as well.

In this notebook, we will explore what we can learn about the lens or transit host system from the complete Roman GBTDS data set. For this group project, we will perform a similar analysis to that done in the main session but focus on the Galactic Center field rather than the lower bulge. Here, extinction is much higher, as well as stellar density, and we have additional Galactic populations to consider like the Nuclear Stellar Disk (NSD) and Nuclear Star Cluster (NSC).


# Data and import set-up

In [ ]:
# Import packages
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy import units as u
from astropy.coordinates import SkyCoord

In [ ]:
# Set up access to the data sets
#
# Before running this cell, as described in the Workshop Google Instructions:
# Make shortcut to the shared drive in your top level Google drive:
# Shared drive: https://drive.google.com/drive/folders/1gJhl7zA3lqkiXx3YnAdgU8Q14tUexLlm?usp=sharing_eil&ts=69fcd6a4$0
#
# You will be prompted to Permit this notebook to access your Google Drive files - Click on "Connect to Google Drive"
# You will then be prompted to Choose an account - click on your preferred Google account
# You will then confirm that Google Drive for desktop wants to access your Google Account - select all and scroll to click "Continue"
# You may get another prompt to allow additional access for this to work - scroll to click "Continue"from google.colab import drive

from google.colab import drive
drive.mount('/content/drive')
#this_dir = '/content/drive/Shareddrives/MULab/research/microlens'
#this_dir += '/Roman/2026 Sagan Activity - Galactic Context/'
this_dir = '/content/drive/MyDrive/2026 Sagan Activity - Galactic Context/'

data_dir = this_dir + 'data/'

# Data set introduction

## Microlensing events table
This is the table of "detected" microlensing events with their best-fit parameters. Some of the columns and units are listed in the table below.

Column | Description | Units
-------|-------------|------
u0     | Projected closest approach | $\theta_E$
t0     | Time of closest approach | days
t_E    | Einstein crossing time | days
pi_E   | Microlensing parallax | $\theta_E$
theta_E | Einstain radius | mas
mag_base_* | Baseline magnitude of target | AB mag
mag_lens_* | Lens magnitude after deblending | AB mag
mag_source_* | Source magnitude after deblending | AB mag
pm_l_cosb_lens | Lens proper motion in R.A. after deblending | mas/yr
pm_b_lens | Lens proper motion in Dec. after deblending | mas/yr
pm_l_cosb_source | Lens proper motion in R.A. after deblending | mas/yr
pm_b_source | Source proper motion in Dec. after deblending | mas/yr


Take a look at the available events and the columns in the table, and select an event to focus on as your target for the rest of the session.



In [ ]:
# Read in the events table
events_table = Table.read(data_dir+'events_gc_table.parquet')

# Limit the number of significant digist for easy display:
for col in events_table.itercols():
    if col.info.dtype.kind == 'f':  # Check if it's a float
        col.info.format = '.5g'    # 3 significant figures

In [ ]:
# Display a preview of the events table
events_table

In [ ]:
# Select your event to explore for this session
# Good ones to try: 2292274
# Try sorting your table for bright events and/or bright lenses to look for other good options
event_star_id = 2292274
event_table_idx = np.argwhere(events_table['star_id']==event_star_id)

## Stars table
This is the catalog of detected "stars" in the subfield we focus on. These "stars" are often blended combinations of multiple stars, with effective combined proper motions and parallaxes measured. Some of the column names and units are in the table below:

Column | Description | Unit
-------|-------------|-----
star_id | Name of Object | -
l_deg | Galactic Longitude | deg
b_deg | Galactic Latitude  | deg
pm_l_cosb | Proper Motion along R.A. | mas/yr
pm_b | Proper Motion along Dec. |mas/yr
pi | Parallax | mas
mag_* | Photometry | AB mag

You can use the star_id column to find your event in this table. The table will show the blended system properties including the source, lens, and any neighbors within 0.09".

In [ ]:
# Read in the stars table
stars_table = Table.read(data_dir+'stars_gc_table.parquet')

# Limit the number of significant digist for easy display:
for col in stars_table.itercols():
    if col.info.dtype.kind == 'f':  # Check if it's a float
        col.info.format = '.5g'    # 3 significant figures

In [ ]:
# Print a preview of the table
stars_table

In [ ]:
stars_table[event_star_id]

# Color-magnitude diagrams
Let's take a look at the stars in our field and put our target in context with a color-magnitude diagram.

What are the structures you see in the CMD? Why are there two branches?

In [ ]:
# Plot some color-magnitude diagrams

## Characterizing your lens and source stars
From the microlensing event fit, we have a "source flux fraction." The is the fraction of the total flux of your blended "star" at baseline that is from your source. For these examples, let's assume that any additional blended light comes from the lens. In reality, there are additional factors that may be involved in making this determination, like detected lens-source separation or astrometric microlensing signals.

What can you say about the age of your lens or host star? What population do you think it belongs to?

In [ ]:
# Plot on your source and lens stars on the CMD

# Now, let's put these into context with some simulations
We can use a Galactic model to better understand the stellar populations shown in the CMDs.

## Simulating the Galaxy
We use PopSyCLE to simulate a realistic population of stars in a sample GBTDS subfield. This generates stars from the major Galactic populations: the bulge, disk, and halo. Each population has a distribution throughout the Galaxy, ages, metallicities, etc. The simulation includes a dustmap to apply realistic extinction.

![galaxy_diagram.png](https://drive.google.com/uc?export=view&id=1Bs-3VImwHTl43kjibM1MPul96ZmphE5f)

In this catalog, populations 3-10 are different components of the disk, 1 is the halo, and 0 is the bulge. The halo can generally be ignored in microlensing toward surveys the bulge (not toward the LMC/SMC!). Population 2 in the Nuclear Stellar Disk which contributes negligibly in the lower bulge fields. However, it is important in the GC here!

Knowing which population your star belongs to can give you hints about things like its age, metallicity, and distance. We can compare our planetary host stars to the model catalog stars to try to disentangle some of these stellar properties.

Let's load up our Galactic model catalog.

In [ ]:
# Load up the simulation table and print the column names / a few example rows
# Note: we downsampled the simulation catalog by a factor of 1/4 since the field is so dense
# You can downsample more as needed for reasonable plotting code execution times
sim_table = Table.read(data_dir+'sim_pops_gc_table.parquet')
print(sim_table.columns)
sim_table[:10]

## From intrinsic properties to observable

First, let's check out the properties of the stars in our catalog in a familiar form: an HR diagram. This shows stellar temperatures versus luminosity, key intrinsic properties that describe their brightness and color.

In [ ]:
# Plot the HR diagram: temperature versus luminosity.

Next, let's move this one step closer to what we'll see with Roman: absolute magnitudes in the Roman filters. First, let's take a look at what the Roman filters (figure from Roman User Documentation / STScI). ![roman_filters_rdox.png](https://drive.google.com/uc?export=view&id=1tlk5PubIluyP-5kTaTdu7ZbmO-9OQ0JE) There are 7 standard-width filters from optical wavelengths through near-infrared (0.5 2.25 microns). The F146 filter, the primary filter for the GBTDS is very wide, covering 1-2 microns.

Let's look at a color-magnitude diagram (CMD), which is similar to the HR diagram, selecting two photometric filters from Roman to use as proxies for luminosity and temperature. Note: we are using absolute magnitudes here, so this is representative of the star's intrinsic properties with no effects from distance or extinction.

In [ ]:
# Plot the absolute CMD in two example Roman filters

When we convert from absolute magnitudes to apparent, we must account for the impact of distance and extinction. Distance simply makes stars appear more or less dim, moving them up or down on this plot. Dust between us and the star also changes the light we see, blocking some fraction of it from reaching us. Shorter wavelengths are more strongly impacted by this effect than longer, so it causes stars to appear both dimmer (extinction) and redder (reddening).

Let's separate the disk and bulge stellar populations now, since these populations have different ages and distributions in distance (and thus extinction). Our next color-magnitude diagram shows the stars' apparent magnitudes, which allows us to compare it to observational data.

In [ ]:
# Separate the bulge and disk populations
sim_table_disk = sim_table[sim_table['pop_id']>=3]
sim_table_nsd = sim_table[sim_table['pop_id']==2]
sim_table_bulge = sim_table[sim_table['pop_id']==0]

In [ ]:
# Make a CMD of simulated stars with color-coding by population

## Putting our target in context

Next, we will compare this observed and simulated CMDs and determine which population the lens and source likely belong to.

Now what age would you infer for your lens or host star? Does it belong to the disk or the bulge? Can you say anything about metallicity?

In [ ]:
# Compare this CMD to the observed one above. Which populations do your lens and source likely belong to?

In [ ]:
# Plot some isochrones over your CMD and estimate the mass and radius of your lens star

The plots above often don't provide enought information to figure out whether the lens is in the disk or the bulge from the CMD alone, but there are additional metrics we can examine.

## Isochrones
An isochrone is a table of current stellar properties for a population of stars with a given age. Each row in the table corresponds to a different initial mass.

Plot some isochrones over your CMD and estimate the mass and radius of your lens star. We will utilize [SPISEA](https://spisea.readthedocs.io/en/latest/) isochrones (pre-computed).

In the SPISEA file names, the first number is the isochrone age in log years. The second number is the extinction in A_Ks (mag). Next is the distance to the stars in parsecs. Next is the metallicity of the stellar population, where 'p' means plus for positive (or zero) metallicity. Finally, we have converted these isochrones from SPISEA's default Vega magnitude system to AB which is standard for Roman.

Our first isochrone here corresponds with a typical bulge/NSD population toward the GC, while the second represents a more nearby disk star with less dust in the foreground and at a younger age.

In [ ]:
iso1 = Table.read(this_dir + 'isochrones/iso_10.00_2.30_08000_p0.00_ABmag.fits')
iso2 = Table.read(this_dir + 'isochrones/iso_7.00_0.50_04000_p0.00_ABmag.fits')
iso = [iso1, iso2]

print(iso1.meta)
print(iso1.columns)

In [ ]:
# Plot some isochrones over your CMD and estimate the mass and radius of your lens star

# Optional: What can we learn from multi-filter photometry?

A Spectral Energy Distribution (SED) is like a low-resolution spectrum of a star, where we have a magnitude measurement in several photometric filters.

Plot an SED for the target. We won't run a full SED fit today, though you could do that for a group project.

From some pre-computed isochrones, we can plot on SEDs of stars with similar F146 mag to roughly estimate potential underlying stellar properties.

Does your star match up well with any SED's from the isochrone? Does one isochrone look like a better fit than the other?

What difficulties can arise in this process, for example if extinction were significantly higher?

In [ ]:
# Plot an SED for the target
# Instead of running a full fit, you can select the stars in our sample isochrones where magnitude in the primary filter is nearest.
# How well does each isochrone star's SED match your star? Is it clear one population is a better match?

# Kinematics

Next, let's examine the kinematics of the target in context.
Plot the proper motions of the simulation catalog stars by population.
Then, add your lens and source to the plot and see how the fit into the populations.

Note that the reported proper motions from the stars table might be blended in cases where the "star" is actually a binary or where neighbor stars are within the beam. In the events table, the proper motions between the source and lens can be decomposed based on the blending factor measured from the microlensing light curve.

In the case of a lumious source and lens, the observed proper motion is given by

$\vec{\mu}_{blended} = \frac{f_S \vec{\mu}_S + f_L \vec{\mu}_L}{f_S + f_L}$

$\vec{\mu}_{blended} = b_{sff} \vec{\mu}_S + (1 - b_{sff}) \vec{\mu}_L$

Tip if you're low on time: you can use the proper motions and parallaxes provided in the events table.

Optional if you have extra time: compare your blend ratio-derived values to those provided in the table. What happens if you have unrelated neighbor stars in your blend?

In [ ]:
# Plot proper motions in the l and b directions for the simulated populations. Where do your source and lens stars lie in this space?

# Let's take a look at parallaxes

Although lens-source separation can give us the individual proper motions of the source and lens, it will be more difficult to get reliable individual parallaxes. Luckily, we have both the blended parallax from the star catalog and the relative lens-source parallax from the microlensing event, so we can disentangle them if we have a measurement of the Einstein ring radius $\theta_E$.

Much like the proper motions, blended parallax is the flux-weighted mean of the lens and source parallax.
The relative parallax is the difference between the lens and source parallax, which is the microlensing parallax times the Einstein ring radius ($\pi_{rel} = \pi_E \times \theta_E$).
Use these plux your source flux fraction to calculate the individual lens and source parallaxes.

In [ ]:
# Calculate the lens and source parallax

Can proper motion and parallax also help clarify which populations the stars belong to?

In [ ]:
# Plot proper motion versus parallax for the simulation populations
# Where does your star lie in this space?

How about magnitude and parallax?



In [ ]:
# Plot parallax versus magnitude

#Star and Planet Masses / Radii

For a microlensing event, we can definitively measure the lens mass if we have the Einstein radius $\theta_{\rm E}$ via astrometric microlensing, lens-source separation, or finite source effects.

It can be derived as $\theta_{\rm E} = \mu_{\rm rel}*t_{\rm E}$ (make sure to check units). If we can measure $\pi_{\rm E}$ from our light curve and/or astrometry, we can then determine the mass of the lens.

$M_{\rm lens}$ = $\theta_{\rm E}$/($\pi_{\rm E} \times$ 8.144 mas/Msun)


In [ ]:
# Calculate theta_E

In [ ]:
# Calculate lens mass

From a planetary event, you can measure q the mass ratio from the light curve. Let's say your q value is $10^{-5}$. What is the mass of your planet in Earth masses? What type of planet is this likely to be given its mass and wide separation?

Note: 1 $M_\odot$ = 333000 $M_\oplus$

In [ ]:
# Calculate planet mass

For a transitting planet, we can use the transit depth to estimate the planetary radius instead.

Transit host star analysis is often simpler because you don't necessarily have a luminous background source contaminating the light of the host star.

# Conclusion + Additional considerations
What were you able to figure out about your host star from its multi-filter photometry and microlensing parameters?

What aspects were ambiguous for this particular event?

What can you then learn about your planet from this?

We won't be able to de-blend the lens and source effectively for every Roman event. What might you still be able to measure in cases where you have, for example, a measurement of finite source events or astrometric microlensing to tell you the Einstein radius?

What was different here than in the main session in the lower bulge field?